In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from PIL import Image
import os
import itertools
import time

# 检查并使用 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 当前使用的计算设备: {device}")
if device.type == 'cpu':
    print("⚠️ 警告：当前没有使用 GPU！请检查 Kaggle 右侧边栏的 Accelerator 设置！")

# ==========================================
# 网络架构搭建 (双生成器 + 双判别器)
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            ResidualBlock(64), ResidualBlock(64),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 3, kernel_size=7, stride=1, padding=3),
            nn.Tanh()
        )
    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 1, kernel_size=4, stride=1, padding=1)
        )
    def forward(self, x):
        return self.model(x)

🚀 当前使用的计算设备: cuda


In [2]:
# ==========================================
# 准备数据集 (请根据你的实际情况修改 DATASET_DIR)
# ==========================================
# 假设你把 monet2photo 文件夹放在了当前目录下：
DATASET_DIR = "./monet2photo" 

dir_A = os.path.join(DATASET_DIR, "trainA") # 真实风景照
dir_B = os.path.join(DATASET_DIR, "trainB") # 莫奈油画

print(f"正在读取风景照: {dir_A}")
print(f"正在读取莫奈画: {dir_B}")

class RealUnpairedDataset(Dataset):
    def __init__(self, dir_A, dir_B, transform):
        self.files_A = sorted([os.path.join(dir_A, f) for f in os.listdir(dir_A) if f.endswith(('jpg', 'png', 'jpeg'))])
        self.files_B = sorted([os.path.join(dir_B, f) for f in os.listdir(dir_B) if f.endswith(('jpg', 'png', 'jpeg'))])
        self.transform = transform
        if len(self.files_A) == 0: raise RuntimeError(f"未找到图片，请检查路径: {dir_A}")

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))

    def __getitem__(self, index):
        item_A = self.transform(Image.open(self.files_A[index % len(self.files_A)]).convert('RGB'))
        item_B = self.transform(Image.open(self.files_B[index % len(self.files_B)]).convert('RGB'))
        return item_A, item_B

# 图像预处理
transform = transforms.Compose([
    transforms.Resize((128, 128)), 
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = RealUnpairedDataset(dir_A, dir_B, transform)
# Kaggle 上 batch_size 设为 4 比较稳妥
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2) 
print(f"✅ 数据集加载成功，共有 {len(dataset)} 对图片样本。")

正在读取风景照: ./monet2photo\trainA
正在读取莫奈画: ./monet2photo\trainB
✅ 数据集加载成功，共有 6287 对图片样本。
